# NeuroFly-Bee Perú — Resultados reales en Google Colab (v19, actualizado en v23)

Un solo comando entrena todas las redes, corre el benchmark y las ablaciones, genera las figuras, empaqueta los pesos y **regenera los números del artículo**, registrando el costo (horas de GPU, R6).

1. *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*.
2. Ejecute las celdas en orden. Viene con `QUICK = False` (corrida completa, ~2–3 h en una T4); la prueba rápida (`QUICK = True`, ~30 min) ya se validó el 4-nov. Los números de la corrida rápida **no** sirven para el artículo.
3. Si Colab se desconecta, vuelva a ejecutar la celda del pipeline: **continúa donde quedó**.
4. Al final se descarga un `.zip` para copiar a la carpeta de la versión.

In [ ]:
!nvidia-smi -L || echo 'Sin GPU: funcionará en CPU, más lento'

## 1. Obtener el código
Con `USE_DRIVE = True` el proyecto y sus resultados viven en `MyDrive/neuroflybee_run`: si Colab se desconecta o se recicla, vuelva a ejecutar las secciones 1, 2 y 3 y el pipeline **continúa donde quedó**. Si el repositorio está en GitHub, ponga su URL; si no, suba el `.zip` de `neuroflybee_colab_v24`.

In [ ]:
import glob, os, zipfile
USE_DRIVE = True   # recomendado: si Colab se desconecta, nada se pierde y el pipeline continúa donde quedó
REPO_URL = ""      # p. ej. https://github.com/<usuario>/neuroflybee.git ; vacío = subir el .zip
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/neuroflybee_run', exist_ok=True)
    os.chdir('/content/drive/MyDrive/neuroflybee_run')
if glob.glob('pyproject.toml') + glob.glob('*/pyproject.toml'):
    print('El proyecto ya está aquí: se reutiliza con sus resultados (no suba el .zip otra vez)')
elif REPO_URL:
    !git clone -q $REPO_URL neuroflybee
else:
    from google.colab import files
    for name in files.upload():
        zipfile.ZipFile(name).extractall('.')
root = os.path.dirname(sorted(glob.glob('pyproject.toml') + glob.glob('*/pyproject.toml'), key=len)[0])
os.chdir(root or '.')
print('Proyecto en', os.getcwd())


## 2. Instalar
Colab ya trae PyTorch.

In [ ]:
!pip install -q -r requirements.txt

## 3. Pipeline completo
Etapas: pruebas → datos → validación del sensor (v21) → mosca → DAgger → abeja → ablación de la abeja (v20) → benchmark → significancia (v22) → figuras → interpretabilidad → entorno → pesos → números. Cada etapa deja su registro en `results/logs/` y su tiempo en `results/v19_costo.csv`.

In [ ]:
import datetime
QUICK = False    # corrida COMPLETA del artículo (3 semillas, 30 escenarios por condición, ~2–3 h); True = prueba rápida
VIDEO = False    # True agrega el video de respaldo con los controladores neuronales
flags = (' --quick' if QUICK else '') + (' --video' if VIDEO else '')

PERU = datetime.timezone(datetime.timedelta(hours=-5))          # hora de Lima (sin horario de verano)
inicio = datetime.datetime.now(PERU)
print('================================================================')
print('INICIO:', inicio.strftime('%Y-%m-%d %H:%M:%S'), '(hora de Lima)')
print('================================================================')
!python scripts/run_pipeline.py --dry-run $flags
!python scripts/run_pipeline.py $flags
fin = datetime.datetime.now(PERU)
dur = fin - inicio
h, rem = divmod(int(dur.total_seconds()), 3600)
m, s = divmod(rem, 60)
print('================================================================')
print('INICIO:  ', inicio.strftime('%Y-%m-%d %H:%M:%S'))
print('FIN:     ', fin.strftime('%Y-%m-%d %H:%M:%S'))
print(f'DURACIÓN: {h} h {m} min {s} s  (reloj de pared de esta celda)')
print('Nota: si Colab se reconectó a mitad, use la línea "Horas totales" de arriba,')
print('      que suma el tiempo de cómputo real de todas las reanudaciones.')
print('================================================================')


## 4. Ver resultados

In [ ]:
import glob, os
if not os.path.exists('pyproject.toml'):   # Colab se reinició: volver a la carpeta del proyecto
    found = sorted(glob.glob('/content/drive/MyDrive/neuroflybee_run/*/pyproject.toml')
                   + glob.glob('/content/*/pyproject.toml') + glob.glob('/content/*/*/pyproject.toml'), key=len)
    if not found:
        raise SystemExit('No se encuentra el proyecto: ejecute de nuevo las secciones 1 y 2 (con USE_DRIVE = True los resultados siguen en Drive).')
    os.chdir(os.path.dirname(found[0]))
print('Proyecto en', os.getcwd())
import pandas as pd
from IPython.display import Image, display
display(pd.read_csv('results/v19_costo.csv'))
display(pd.read_csv('results/v14_tabla_principal.csv'))
for extra in ['results/v20_ablacion_abeja_resumen.csv', 'results/v22_significancia.csv']:
    if os.path.exists(extra):
        display(pd.read_csv(extra))
for f in sorted(glob.glob('results/figures/v9_*.png') + glob.glob('results/figures/v1[0-5]_*.png') + glob.glob('results/figures/v2[0-2]_*.png')):
    print(f); display(Image(f, width=900))

## 5. Artículo con los números nuevos (opcional)
El artículo toma cada número de `paper/generado/numeros.tex`. Lo que sigue en rojo aún falta.

In [ ]:
import glob, os
if not os.path.exists('pyproject.toml'):   # Colab se reinició: volver a la carpeta del proyecto
    found = sorted(glob.glob('/content/drive/MyDrive/neuroflybee_run/*/pyproject.toml')
                   + glob.glob('/content/*/pyproject.toml') + glob.glob('/content/*/*/pyproject.toml'), key=len)
    if not found:
        raise SystemExit('No se encuentra el proyecto: ejecute de nuevo las secciones 1 y 2 (con USE_DRIVE = True los resultados siguen en Drive).')
    os.chdir(os.path.dirname(found[0]))
print('Proyecto en', os.getcwd())
!apt-get -qq install -y latexmk texlive-latex-extra texlive-lang-spanish > /dev/null
!cd paper && latexmk -pdf -interaction=nonstopmode -quiet main.tex; ls -la paper/main.pdf

## 6. Publicar los pesos (enlace obligatorio, §3.3)
Opción A: Hugging Face (`huggingface-cli login` con un token de escritura). Opción B: subir `neuroflybee_pesos.zip` a Google Drive y compartir el enlace.

In [ ]:
import glob, os
if not os.path.exists('pyproject.toml'):   # Colab se reinició: volver a la carpeta del proyecto
    found = sorted(glob.glob('/content/drive/MyDrive/neuroflybee_run/*/pyproject.toml')
                   + glob.glob('/content/*/pyproject.toml') + glob.glob('/content/*/*/pyproject.toml'), key=len)
    if not found:
        raise SystemExit('No se encuentra el proyecto: ejecute de nuevo las secciones 1 y 2 (con USE_DRIVE = True los resultados siguen en Drive).')
    os.chdir(os.path.dirname(found[0]))
print('Proyecto en', os.getcwd())
HF_REPO = ''   # p. ej. 'usuario/neuroflybee'
if HF_REPO:
    !pip install -q huggingface_hub
    !python scripts/package_weights.py upload --repo $HF_REPO

## 7. Descargar todo

In [ ]:
import glob, os
if not os.path.exists('pyproject.toml'):   # Colab se reinició: volver a la carpeta del proyecto
    found = sorted(glob.glob('/content/drive/MyDrive/neuroflybee_run/*/pyproject.toml')
                   + glob.glob('/content/*/pyproject.toml') + glob.glob('/content/*/*/pyproject.toml'), key=len)
    if not found:
        raise SystemExit('No se encuentra el proyecto: ejecute de nuevo las secciones 1 y 2 (con USE_DRIVE = True los resultados siguen en Drive).')
    os.chdir(os.path.dirname(found[0]))
print('Proyecto en', os.getcwd())
!zip -q -r resultados_v24.zip checkpoints results paper/tablas paper/generado requirements-lock.txt neuroflybee_pesos.zip -x 'results/logs/*'
from google.colab import files
files.download('resultados_v24.zip')